# Module 02 — Write Problems (Colab)

**The story.** NovaBridge runs an AI agent whose job is to charge a client their
$2,500 quarterly advisory fee. Like every agent, it **retries** when something
hiccups (a timeout, a dropped connection) — so the "charge the fee" step can run
more than once. If you're not careful, the client is billed **twice**.

**Why this is an *agent* problem, not just an old one.** Double-charges on retries
are a classic problem — payment systems solved it years ago. But agents take away
the two things that made it easy: the retry no longer sends the *exact same request*
(the model rewrites its output every run), and the most tempting thing to identify a
charge by — the text the model wrote — is the one thing that *isn't* stable. So the
old problem comes back wearing a disguise, and the obvious fixes quietly fail.

**What you'll do.** Run the agent, watch it double-charge, try the two "obvious"
fixes and see each one fail, then land the fix that actually holds. You only ever
edit one file: `your_fix.py`.

## 1. Set up  *(run once, ~2 min)*

📄 **What this cell does — get the code onto this machine.**

Colab gives you a fresh, empty Linux computer in the cloud. This cell moves into its
working folder (`/content`), deletes any old copy of the project (`rm -rf repo`) so
you always start clean, downloads the workshop code from GitHub (`git clone`), and
steps into that folder. After this, all the lab files are on the machine.

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

📄 **What this cell does — install the database and Python libraries.**

`setup.sh` is a one-shot installer: it sets up a real **PostgreSQL database** inside
this machine and installs the Python packages the labs need. `SKIP_OLLAMA=1` skips
downloading the AI model itself — on the participant path we *replay recorded model
answers* instead (faster, and identical for everyone). The second line is a safety net
that makes sure the database driver (`psycopg`) is installed even on picky Colab runtimes.

👀 **What you'll see:** a few minutes of install logs ending in `✅ SETUP COMPLETE`.
Some lines may look like warnings — that's normal as long as you reach SETUP COMPLETE.

In [ ]:
!SKIP_OLLAMA=1 bash setup.sh
# Failsafe: ensure Python deps are installed even on PEP 668 runtimes.
!python -m pip install --break-system-packages -q "psycopg[binary]==3.2.9"

📄 **What this cell does — tell the code which model and database to use.**

These are two settings the code reads. `NOVA_LLM = 'frozen'` means "don't call a live
model — replay the real answers we recorded earlier," which keeps the lab fast and
100% reproducible. `DATABASE_URL` points the code at the local Postgres database you
just set up. (The very last cell of the notebook shows how to switch to a live model
if you want to.)

In [ ]:
import os
os.environ['NOVA_LLM'] = 'frozen'  # replay recorded real model outputs (deterministic)
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'

## 2. Meet the agent

📄 **What this cell does — wire up the agent and load the client's document.**

An "agent" here isn't magic — it's just ordinary code you can call. This cell imports
its parts: the thing that reads a document, the **payment gateway** (a stand-in for a
real card processor), the model interface, and the database. It then connects to the
database and loads client *alpha*'s billing note.

👀 **What you'll see:** the text of the billing note the agent will work from.

In [ ]:
import sys, importlib, hashlib
sys.path.insert(0, 'modules/02_write_path')

from nova.agent import load_document, payment_memo_prompt
from nova.effects import PaymentGateway
from nova.llm import get_llm
from nova.store import get_store
import your_fix

llm = get_llm()
store = get_store(); store.init_schema()
doc = load_document('alpha', 'billing_instruction.md')
print(doc)

📄 **What this cell does — run the agent once.**

The agent reads the billing note and writes a one-line **payment memo** (a short
human-readable description of the charge). This is the agent "thinking" — producing
text from the document.

👀 **What you'll see:** the memo it wrote on attempt 1.

In [ ]:
memo_1 = llm.complete(payment_memo_prompt('alpha', 'Q1-2026', doc, 1))
print('attempt 1:', memo_1)

📄 **What this cell does — run it again, as if the agent retried.**

Same client, same fee, same inputs — but we run the agent a second time and compare
the two memos. **This is the key moment of the whole lab:** the wording comes out
*different*. AI output is non-deterministic. Remember this — the second trap depends
entirely on it.

👀 **What you'll see:** a second, differently-worded memo, and `same memo? False`.

In [ ]:
memo_2 = llm.complete(payment_memo_prompt('alpha', 'Q1-2026', doc, 2))
print('attempt 2:', memo_2)
print('same memo?', memo_1 == memo_2)

🔑 **Two things to hold onto before the labs:**

1. The **payment gateway** is the outside world — every call to it *moves real money*
   and cannot be undone. It is *not* your database.
2. Your own `charges` table has a **unique key**, so it refuses to store the same
   charge row twice. Keep an eye on the difference between "my records" and "the card."

## 3. The first "obvious" fix: lean on the database's unique key

The instinct: *my `charges` table already rejects duplicates, so I'll just charge the
card and then record the row — the table will handle the duplicate.* Let's test that
instinct.

### ✋ Predict #1

We charge the fee once, then retry it. **Before running:** how many rows end up in the
`charges` table, and how many times does the card actually get charged? Write down a
guess.

📄 **What this cell does — run the "charge, then record" approach across a retry.**

`table_key_charge` does exactly what the instinct says, in this order: (1) charge the
card, then (2) record a row using a unique key. `reset_demo()` clears any leftover data
so we start fresh. We then run it **twice** (the agent + its retry) and count both the
database rows and the real card charges.

👀 **What you'll see:** `rows in charges table: 1` but `times the card was charged: 2`.
The table deduped — the card did not.

In [ ]:
def table_key_charge(gateway, store, client_id, period, amount, memo):
    key = hashlib.sha256(f'{client_id}|{period}'.encode()).hexdigest()
    gateway.charge(client_id, amount, memo)      # irreversible effect fires first
    store.record_charge(key, client_id, amount)  # unique key -> only one row survives

store.reset_demo(); gateway = PaymentGateway()
for memo in (memo_1, memo_2):   # the agent runs, then retries
    table_key_charge(gateway, store, 'alpha', 'Q1-2026', 2500, memo)

print('rows in charges table:', len(store.get_charges('alpha')))
print('times the card was charged:', len(gateway.charges_for('alpha')))

📄 **What this cell does — look inside the real database.**

This runs a plain SQL query against the actual Postgres database to show the `charges`
table. It proves the previous result isn't a trick of the Python code — your records
genuinely show a single, clean charge.

👀 **What you'll see:** exactly one row for `alpha`.

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, amount FROM charges;"

### The aha (1 of 2)

Your database looks perfect: **one** row. But the client was charged **twice** — the
money moved *before* the unique key ever got a say. A database constraint protects
*your records*; it cannot *un-charge a card*.

> **Lesson 1:** For anything irreversible, the guard has to sit **before** the effect —
> check "did I already do this?" *first*, and only then act.

## 4. Guard the effect — your first attempt at the fix

Now you move the check ahead of the charge. But that raises a new question: *how do you
recognise "this fee" so you can tell it's already been charged?* You need an ID for the
charge. The tempting choice: the memo the agent wrote.

📄 **What this cell does — show the file you're about to edit.**

`your_fix.py` is the *only* file you change in this lab. This prints its current
contents so you can see the starting point before you overwrite it in the next cell.

In [ ]:
print(open('modules/02_write_path/your_fix.py').read())

📄 **What this cell does — write your first fix (guard before charging, keyed on the memo).**

The `%%writefile` line at the top means "save everything below into that file." This
version does the right structural thing — it calls `already_charged(...)` and returns
**before** touching the gateway if the fee was already charged — but it builds the key
from the **memo text**. Reasonable-looking. Watch what happens.

In [ ]:
%%writefile modules/02_write_path/your_fix.py
import hashlib


def charge_key(client_id, billing_period, memo):
    return hashlib.sha256(memo.encode('utf-8')).hexdigest()  # key on the memo


def charge_client_fee(gateway, store, client_id, billing_period, amount, memo):
    key = charge_key(client_id, billing_period, memo)
    if store.already_charged(key):
        return                                # guard BEFORE the effect
    gateway.charge(client_id, amount, memo)
    store.record_charge(key, client_id, amount)

📄 **What this cell does — test the *easy* retry (the same memo replayed).**

`importlib.reload` loads the file you just wrote. Then we simulate a retry where the
*same* memo comes back (`memo_1` both times) and count the card charges. Finally we run
the automated test for this exact case.

👀 **What you'll see:** `times the card was charged: 1` and the test **passes**. Looks
fixed... but notice we cheated slightly by replaying the *same* memo.

In [ ]:
importlib.reload(your_fix)
store.reset_demo(); gateway = PaymentGateway()
for _ in range(2):   # the retry replays the SAME memo
    your_fix.charge_client_fee(gateway, store, 'alpha', 'Q1-2026', 2500, memo_1)

print('times the card was charged:', len(gateway.charges_for('alpha')))
!python -m pytest modules/02_write_path/test_write.py::test_replayed_retry_charges_the_client_only_once -q

### ✋ Predict #2

A real retry usually isn't a clean replay — the agent re-runs and, as you saw at the
top, the model **rewrites the memo** (`memo_1` vs `memo_2`). Same fee, different memo.
**Before running:** with the charge keyed on the memo, how many times is the card
charged now?

📄 **What this cell does — run the *realistic* retry (a rewritten memo).**

Identical to before, except the retry uses the *different* memo (`memo_2`) — exactly
what happens when a non-deterministic agent re-runs. Count the card charges.

👀 **What you'll see:** `times the card was charged: 2`. The double charge is back.

In [ ]:
store.reset_demo(); gateway = PaymentGateway()
for memo in (memo_1, memo_2):   # the agent re-ran; the model rewrote the memo
    your_fix.charge_client_fee(gateway, store, 'alpha', 'Q1-2026', 2500, memo)

print('times the card was charged:', len(gateway.charges_for('alpha')))

📄 **What this cell does — run the full test suite for this lab.**

Two tests run: the replayed retry (which your memo-key fix passes) and the regenerated
retry (which it fails). Seeing one pass and one fail is the whole point — a fix can look
correct and still be broken by the agent's non-determinism.

👀 **What you'll see:** `1 failed, 2 passed` (the failure is the regenerated-retry test).

In [ ]:
!python -m pytest modules/02_write_path/test_write.py -q

### The aha (2 of 2)

The memo is the agent's **output**, and the model rewrites it every run — so the "same"
fee got two different keys and slipped past your guard. This is the distinctively
*agentic* trap: you reached for the model's output as an identifier, and the model's
output isn't stable.

> **Lesson 2:** Key on the fee's **intent** — *which client, which period* — the one
> thing that stays the same across every retry. Never key on what the model produced.

📄 **What this cell does — write the real fix (key on intent).**

Same structure as before — still guarding *before* the charge — but `charge_key` now
ignores the memo entirely and is built from `client_id` + `billing_period`. That key is
identical on every attempt, so a retry is recognised as the same fee.

In [ ]:
%%writefile modules/02_write_path/your_fix.py
import hashlib


def charge_key(client_id, billing_period, memo):
    # key on INTENT (client + period); ignore the memo, it changes every run
    return hashlib.sha256(f'{client_id}|{billing_period}'.encode('utf-8')).hexdigest()


def charge_client_fee(gateway, store, client_id, billing_period, amount, memo):
    key = charge_key(client_id, billing_period, memo)
    if store.already_charged(key):
        return
    gateway.charge(client_id, amount, memo)
    store.record_charge(key, client_id, amount)

📄 **What this cell does — confirm both tests now pass.**

Reload your fixed file and run the full suite again.

👀 **What you'll see:** `3 passed` — both retry kinds are now safe.

In [ ]:
importlib.reload(your_fix)
!python -m pytest modules/02_write_path/test_write.py -q

📄 **What this cell does — show the full before/after in one view.**

`compare.py` runs both retry scenarios (replayed and regenerated) against the naive
code and against your fix, and prints a small table so you can see the whole story at
a glance.

In [ ]:
!python modules/02_write_path/compare.py

### The takeaway

Doing something irreversible *exactly once* with an agent needs **both** halves:

1. **Guard before the effect** — the classic part you already knew.
2. **Key on intent, never on the model's output** — the new part agents force on you.

That's what "data infrastructure for agents" really means: not new database technology,
but the same fundamentals made *load-bearing*, because your producer is non-deterministic
and retries constantly.

### Optional: run the real local model

Everything above replayed recorded outputs so the double charge is reproducible. To watch
the same design play out against a live local model, install Ollama and set `NOVA_LLM=ollama`
(see `SETUP.md`). The design lesson is identical — only the memo wording changes.